# A Duality Principle for Optimal Transport — experiments

Companion code for *"A Duality Principle for Optimal Transport — Bridging
Geometric and Computational Perspectives."* It runs the certified Kantorovich
duality-gap framework `G = U − L` on **real public datasets** and reproduces
the paper's **synthetic** (closed-form Bures–Wasserstein) figures — end to
end, from a fresh `git clone`.

## How to run on Colab (A100)
1. **Runtime ▸ Change runtime type ▸ A100 GPU** (any GPU works; A100 is fastest; CPU also works).
2. **Runtime ▸ Run all.**
3. Every result (metrics JSON, figures PDF+PNG, run log, manifest) is written to
   `results/` **and** packaged into **`ot_duality_results.zip`**, which downloads
   automatically at the end.

Full run is ≈60–110 min on an A100 (now also includes a neural-OT (ICNN)
frontier point, a GPU memory/OOM stress test out to n=10⁶, and single-cell
nonlinear/raw-gene-space and biological gap-vs-accuracy studies), well under
Colab's session cap. Set `FAST = True` in the config cell for a ~4-minute smoke test.

All Sinkhorn solves run in the **log domain** (`method="sinkhorn_log"`), so the
certified bounds `L ≤ OT* ≤ U` stay valid even at the smallest `eps` swept. The
memory stress test adapts to the actual GPU and catches out-of-memory gracefully,
so it never crashes the session.

## 1 · Clone the repository

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/kagtgi/DualityOT.git"
REPO_DIR = "DualityOT"

# Optional: for a private fork, set a token first -- os.environ["GITHUB_TOKEN"] = "ghp_..."
_tok = os.environ.get("GITHUB_TOKEN", "").strip()
_url = REPO_URL
if _tok and _url.startswith("https://github.com/"):
    _url = _url.replace("https://github.com/", f"https://{_tok}@github.com/")

# Run from inside the repo. If the toolkit already sits next to this notebook
# (a local checkout) we are in place; otherwise clone and cd into it.
if not os.path.exists("otkit.py"):
    if not os.path.isdir(REPO_DIR):
        print("Cloning", REPO_URL, "...")
        subprocess.run(["git", "clone", "--depth", "1", _url], check=True)
    os.chdir(REPO_DIR)

sys.path.insert(0, os.getcwd())
assert os.path.exists("otkit.py"), f"repo layout not found in {os.getcwd()}"
print("Working directory:", os.getcwd())
print("Files:", [f for f in sorted(os.listdir()) if not f.startswith(".")])

## 2 · Install dependencies

In [ ]:
# Colab already ships a CUDA build of torch/torchvision plus numpy, scipy,
# scikit-learn and matplotlib. Install ONLY the extras, so the GPU build of
# torch is never replaced by a CPU wheel.
import sys, subprocess

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# core extras (safe; no torch / numpy churn)
pip_install("POT>=0.9.0", "scikit-image")
# single-cell stack is optional and heavier; best-effort -- a synthetic fallback
# is used if it is unavailable, so the run never blocks on it.
pip_install("scanpy", "anndata")

import ot
print("POT", ot.__version__)
try:
    import torch
    print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except Exception as e:
    print("torch unavailable -> CPU only:", e)

## 3 · Configuration & setup

In [ ]:
import os, time, json, platform, random
import numpy as np
import ot
import otkit, datasets, experiments_synth, experiments_real, make_figures

FAST    = False    # True -> ~3-min smoke test (smaller sweeps); False -> full run
USE_GPU = True     # uses the A100 when present; falls back to CPU automatically
SEED    = 0

random.seed(SEED); np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except Exception:
    pass

OUT = "results"
for sub in ["metrics", "figures", "logs"]:
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)

LOGLINES = []
def log(*a):
    s = " ".join(str(x) for x in a); print(s); LOGLINES.append(s)

def _json_default(o):
    if isinstance(o, np.integer):  return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.ndarray):  return o.tolist()
    return str(o)

log("GPU:", otkit.gpu_info())
log("versions: POT", ot.__version__, "| numpy", np.__version__,
    "| python", platform.python_version())
log("config: FAST =", FAST, "| USE_GPU =", USE_GPU)

## 4 · Synthetic experiments (reproduce the paper's figures)
Closed-form Bures–Wasserstein ground truth (strengthened sweeps):
frontier n=2000 (now incl. an uncertified **neural-OT / ICNN** point),
gap-geometry n=3000 (ε to 0.0005), dimension n=3000 (500 proj),
scaling Sinkhorn to n=32k + sliced to n=100k, hybrid-dim n=2000 (d≤32), two-moons n=2000,
and a GPU **memory/OOM stress test** (dense Sinkhorn wall vs. one-sided surrogates to n=10⁶).

In [ ]:
t0 = time.time()
synth = experiments_synth.run_synth(log=log, fast=FAST, use_gpu=USE_GPU)
T_SYNTH = time.time() - t0
json.dump(synth, open(f"{OUT}/metrics/results_synth.json", "w"), default=_json_default)
log(f"[synthetic suite done in {T_SYNTH:.0f}s] -> {OUT}/metrics/results_synth.json")

## 5 · Real-data experiments
Single-cell PBMC, MNIST↔USPS and color transfer (n=2000 each): frontier, composed
certificate, and the dimensional deficit. For single-cell we additionally test the
deficit in a **nonlinear** diffusion-map embedding and in the **raw native gene
space** (no PCA), plus a **biological** gap-vs-accuracy study (cell-type label
transfer under a controlled batch shift) and a UMAP transport-quiver visual.
Also: large-N GPU scaling (to n=32k on MNIST) and the MNIST→USPS gap-vs-accuracy
sweep. (All loaders fall back gracefully if a download/dependency is unavailable,
so the run never crashes.)

In [ ]:
t0 = time.time()
real = experiments_real.run_real(log=log, fast=FAST, use_gpu=USE_GPU)
T_REAL = time.time() - t0
log(f"[real-data suite done in {T_REAL:.0f}s]")

## 6 · Generate all figures (PDF + PNG)

In [ ]:
made = make_figures.generate_all(synth, real, f"{OUT}/figures", font_dir=".")
log("figures written:", made)

# color-transfer image arrays -> npz, then strip them from the JSON-bound copy
cimg = real.pop("_color_imgs", None)
if cimg is not None:
    np.savez_compressed(f"{OUT}/color_images.npz",
                        **{k: np.asarray(v) for k, v in cimg.items()})
# single-cell transport-quiver arrays are only needed for the figure (already
# drawn above); drop them so the metrics JSON stays small.
real.pop("_singlecell_viz", None)
json.dump(real, open(f"{OUT}/metrics/results_real.json", "w"), default=_json_default)
log("metrics saved ->", f"{OUT}/metrics/")

## 7 · Manifest, package & download

In [ ]:
import shutil

# pull a few headline numbers into the manifest for quick inspection
_ms = synth.get("memory_stress", {})
_neural = next((r for r in synth.get("frontier", {}).get("rows", [])
                if r.get("family") == "neural"), None)

manifest = dict(
    title="A Duality Principle for OT — experiment results",
    fast=FAST, use_gpu=USE_GPU, gpu=otkit.gpu_info(),
    timings_sec=dict(synthetic=round(T_SYNTH, 1), real=round(T_REAL, 1)),
    datasets={k: v["meta"] for k, v in real["datasets"].items()},
    scaling=real.get("scaling_meta"),
    downstream=real.get("da_accuracy", {}).get("meta"),
    downstream_bio=real.get("sc_da_accuracy", {}).get("meta"),
    memory_stress=dict(oom_at=_ms.get("oom_at"), gpu_total_gb=_ms.get("gpu_total_gb"),
                       light_max_n=(_ms.get("light", [{}])[-1].get("n") if _ms.get("light") else None)),
    neural_ot=(dict(rel_err_disc=_neural.get("rel_err_disc"), time=_neural.get("time"))
               if _neural else "not converged / skipped"),
    figures=made,
    python=platform.python_version(), pot=ot.__version__, numpy=np.__version__)
json.dump(manifest, open(f"{OUT}/manifest.json", "w"), indent=2, default=_json_default)
open(f"{OUT}/logs/run.log", "w").write("\n".join(LOGLINES))

shutil.make_archive("ot_duality_results", "zip", OUT)
print("\nWROTE ot_duality_results.zip  (and the results/ directory)\n")
print(json.dumps(manifest, indent=2, default=_json_default))

try:
    from google.colab import files
    files.download("ot_duality_results.zip")
except Exception:
    print("\n(not on Colab; grab ot_duality_results.zip or the results/ folder "
          "from the file panel)")